# Resultados iniciales
## Tema 13: Identificación de menciones de entidades biomédicas en resúmenes de investigación

Integrantes:
- José Ricardo Méndez González, 21289
- Sara María Pérez Echeverría, 21371
- Emily Elvia Melissa Pérez Alarcón, 21385
- Adrian Fulladolsa Palma, 21592

In [3]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import pandas as pd
import nltk
import re

ModuleNotFoundError: No module named 'nltk'

### Preprocesamiento

In [2]:
# download necessary nltk resources
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/melissa/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/melissa/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
# function to clean and preprocess text
def preprocessText(text):
    # convert to lowercase
    text = text.lower()
    
    # remove non-alphabetic characters
    text = re.sub(r'[^a-z\s]', '', text)
    
    # tokenize by spaces
    tokens = text.split()
    
    # remove stopwords
    stopWords = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stopWords]
    
    # lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

In [4]:
# path of the data
dataPath = '../data/'

# load data
abstractsTrain = pd.read_csv(dataPath + 'abstracts_train.csv', on_bad_lines='skip', delimiter='\t')
entitiesTrain = pd.read_csv(dataPath + 'entities_train.csv', on_bad_lines='skip', delimiter='\t')
relationsTrain = pd.read_csv(dataPath + 'relations_train.csv', on_bad_lines='skip', delimiter='\t')
abstractsTest = pd.read_csv(dataPath + 'abstracts_test.csv', on_bad_lines='skip', delimiter='\t')

In [5]:
# apply the preprocessing function to the abstract text column
abstractsTrain['cleanAbstract'] = abstractsTrain['abstract'].apply(preprocessText)
abstractsTest['cleanAbstract'] = abstractsTest['abstract'].apply(preprocessText)

# show the cleaned texts
print(abstractsTrain[['abstract', 'cleanAbstract']].head())
print(abstractsTest[['abstract', 'cleanAbstract']].head())

                                            abstract  \
0  We report on a new allele at the arylsulfatase...   
1  Classical phenylketonuria is an autosomal rece...   
2  The metabolism of the cardioselective beta-blo...   
3  Previous experiments in this laboratory have s...   
4  Eighty unrelated individuals with Duchenne mus...   

                                       cleanAbstract  
0  report new allele arylsulfatase arsa locus cau...  
1  classical phenylketonuria autosomal recessive ...  
2  metabolism cardioselective betablocker metopro...  
3  previous experiment laboratory shown microinje...  
4  eighty unrelated individual duchenne muscular ...  
                                            abstract  \
0  The effect of induced hypertension instituted ...   
1  A linkage study in 30 Becker muscular dystroph...   
2  The effects of a 6-hour infusion with haloperi...   
3  Fragments of the adrenoleukodystrophy (ALD) cD...   
4  The ability to scan a large gene rapidly and a... 

In [6]:
# group entities by abstract_id
entities_grouped = entitiesTrain.groupby('abstract_id')['type'].apply(list).reset_index()

# merge the cleaned abstracts with their corresponding entities by 'abstract_id'
data_merged = pd.merge(abstractsTrain, entities_grouped, how='inner', on='abstract_id')

### Implementación de modelos

In [7]:
from sklearn.model_selection import train_test_split

# dataset splitting, 80% for training and 20% for testing
X_train, X_val, y_train, y_val = train_test_split(
    data_merged['cleanAbstract'], data_merged['type'], test_size=0.2, random_state=42,
)

# sizes of the splits
print(f'Training set size: {len(X_train)}')
print(f'Validation set size: {len(X_val)}')

Training set size: 320
Validation set size: 80


#### Long Short-Term Memory Networks (LSTM)

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

# Convert text to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)

# Pad sequences to ensure uniform input size
max_length = max(len(seq) for seq in X_train_seq)
X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_length, padding='post')

# Define the LSTM model
model = Sequential([
    Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=128, input_length=max_length),
    LSTM(units=128, return_sequences=False),
    Dense(units=len(set(data_merged['type'].explode())), activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train_pad, y_train, validation_data=(X_val_pad, y_val), epochs=10, batch_size=32)

#### Support Vector Machines (SVM)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# Vectorize the text data using TF-IDF
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

# Define the SVM model
svm_model = SVC(kernel='linear')

# Train the model
svm_model.fit(X_train_tfidf, y_train)

# Predict on the validation set
y_pred = svm_model.predict(X_val_tfidf)

# Print the classification report
print(classification_report(y_val, y_pred))

#### Graph Convolutional Networks (GCN)

#### Bidirectional Encoder Representations from Transformers (BERT)